# Självstudier – Föreläsning 5: Regression och första ML-modellen

Nu går kursen från dataanalys till maskininlärning. Målet är att låta datorn lära sig att **gissa priset på en bostad** utifrån exempel på bostäder som redan har sålts.

**Del 1 – För hand.** Du gör det själv, med sådant du redan kan: funktioner, `for`-loopar och pandas. Syftet är att du ska se att maskininlärning inte är magi.

**Del 2 – Med ett färdigt verktyg.** Du gör samma sak med biblioteket scikit-learn och tar reda på om gissningarna håller även för bostäder som datorn aldrig har sett.

De nya begreppen förklaras längs vägen, och i slutet finns en sammanfattning av hela arbetsflödet.

# Del 1: En modell för hand

## 1. Från att beskriva till att förutsäga

I föreläsning 4 frågade vi: *Vad berättar datan?*

Nu frågar vi: *Kan data användas för att förutsäga ett värde vi inte känner till?*

I vanlig programmering skriver vi reglerna själva:

**regler + indata → svar**

I maskininlärning ger vi datorn många **exempel** – indata tillsammans med rätt svar – och låter den hitta regeln:

**indata + svar → regler**

Det är användbart när vi har gott om exempel men har svårt att skriva reglerna själva, till exempel bostadens egenskaper → pris.

## 2. Ett litet bostadsdataset

Vi skapar syntetiska data så att vi känner till ungefär hur datan uppstått. Priset påverkas av storlek, antal rum och ålder – plus lite slumpmässig variation. Det är samma dataset som på föreläsningen och i laboration 5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 140

housing = pd.DataFrame({
    "size_m2": rng.uniform(35, 160, n),
    "rooms": rng.integers(1, 6, n),
    "age_years": rng.integers(0, 80, n),
})

housing["price_ksek"] = (
    500
    + 38 * housing["size_m2"]
    + 160 * housing["rooms"]
    - 7 * housing["age_years"]
    + rng.normal(0, 350, n)
)

housing.head()

Maskininlärning börjar fortfarande med vanlig dataanalys. Titta först på sambandet mellan storlek och pris.

In [ ]:
housing.plot.scatter(x="size_m2", y="price_ksek")
plt.xlabel("Storlek (m²)")
plt.ylabel("Pris (tusen kr)")
plt.title("Bostadsdata")
plt.show()

### Prova själv

Titta på diagrammet. En bostad på **90 m²** kommer ut till försäljning. Vad skulle du gissa att den kostar? Hur kom du fram till gissningen?

## 3. En modell är en funktion

Troligen använde du en regel i stil med *"ungefär så här många tusen kronor per kvadratmeter"*. En sådan regel kan vi skriva som en funktion, precis som i föreläsning 3:

In [ ]:
def gissa_pris(storlek):
    return 45 * storlek


gissa_pris(90)

Talet 45 valde vi själva. Vi gör det till en **parameter** och lägger till en konstant, så att linjen inte måste gå genom origo:

**pris ≈ lutning × storlek + konstant**

In [ ]:
def linjar_modell(storlek, lutning, konstant):
    return lutning * storlek + konstant


linjar_modell(90, lutning=45, konstant=0)

Funktionen fungerar också på en hel kolumn på en gång, precis som beräkningarna på kolumner i föreläsning 4. Det använder vi i en hjälpfunktion som ritar datan tillsammans med en linje:

In [ ]:
def rita_linje(lutning, konstant):
    plt.scatter(housing["size_m2"], housing["price_ksek"], alpha=0.6)
    storlekar = np.linspace(30, 165, 100)
    plt.plot(storlekar, linjar_modell(storlekar, lutning, konstant), color="red")
    plt.xlabel("Storlek (m²)")
    plt.ylabel("Pris (tusen kr)")
    plt.title(f"lutning = {lutning}, konstant = {konstant}")
    plt.show()


rita_linje(lutning=45, konstant=0)

### Prova själv

Anropa `rita_linje` med några egna värden på `lutning` och `konstant`. Kan du hitta en linje som ser ut att passa datan bättre?

In [ ]:
# Prova egna värden här
rita_linje(lutning=35, konstant=1000)

## 4. Hur mycket missar linjen?

Att titta fungerar för ett par förslag, men om vi vill jämföra hundratals linjer behöver vi **ett tal** som beskriver hur bra en linje är.

För varje bostad kan vi räkna ut felet:

**fel = gissning − verkligt pris**

In [ ]:
housing["gissning"] = linjar_modell(housing["size_m2"], lutning=45, konstant=0)
housing["fel"] = housing["gissning"] - housing["price_ksek"]

housing[["size_m2", "price_ksek", "gissning", "fel"]].head()

I diagrammet nedan är varje grått streck felet för en bostad. För en perfekt linje skulle alla streck ha längden noll.

In [ ]:
plt.scatter(housing["size_m2"], housing["price_ksek"], alpha=0.6, label="verkligt pris")
plt.plot(housing["size_m2"], housing["gissning"], color="red", label="gissning")
plt.vlines(housing["size_m2"], housing["price_ksek"], housing["gissning"],
           color="gray", alpha=0.5, label="fel")
plt.xlabel("Storlek (m²)")
plt.ylabel("Pris (tusen kr)")
plt.legend()
plt.show()

Hur gör vi 140 fel till ett enda tal? Den första idén är att ta medelvärdet – men jämför med medelvärdet av felens **absolutbelopp**:

In [ ]:
print("Medelvärde av felen:  ", housing["fel"].mean())
print("Medelvärde av |felen|:", housing["fel"].abs().mean())

Linjen gissar för högt på vissa bostäder och för lågt på andra. Positiva och negativa fel tar därför ut varandra, och medelvärdet blir nästan noll trots att linjen ofta missar med flera hundra tusen kronor.

Därför använder vi felets storlek, `abs()`. Måttet kallas **MAE – Mean Absolute Error** och svarar ungefär på:

> Hur mycket missar modellen i genomsnitt?

Enheten blir samma som för priset (tusen kronor), och lägre är bättre. Vi gör en funktion av det:

In [ ]:
def medelfel(lutning, konstant):
    gissning = linjar_modell(housing["size_m2"], lutning, konstant)
    return (gissning - housing["price_ksek"]).abs().mean()


print("lutning 45, konstant 0:   ", medelfel(45, 0))
print("lutning 35, konstant 1000:", medelfel(35, 1000))

### Prova själv

Prova några olika värden. Hur lågt medelfel kan du få för hand?

In [ ]:
# Prova egna värden här
medelfel(lutning=40, konstant=500)

## 5. Låt datorn prova – träning

Att prova för hand går långsamt, men vi kan låta en `for`-loop göra jobbet. Först håller vi konstanten fast på 500 och provar alla lutningar mellan 20 och 60:

In [ ]:
lutningar = []
medelfelen = []

for lutning in range(20, 61):
    lutningar.append(lutning)
    medelfelen.append(medelfel(lutning, konstant=500))

kandidater = pd.DataFrame({"lutning": lutningar, "medelfel": medelfelen})

kandidater.plot(x="lutning", y="medelfel", legend=False)
plt.xlabel("Lutning")
plt.ylabel("Medelfel (tusen kr)")
plt.title("Hur felet beror på lutningen")
plt.show()

kandidater.sort_values("medelfel").head(3)

Kurvan har ett tydligt minimum: där ligger den bästa lutningen (givet att konstanten är 500).

Nu provar vi **alla kombinationer** av lutning och konstant. Vi lägger en loop *inuti* en annan: för varje lutning provar vi alla konstanter.

In [ ]:
lutningar = []
konstanter = []
medelfelen = []

for lutning in range(20, 61):
    for konstant in range(-500, 2001, 50):
        lutningar.append(lutning)
        konstanter.append(konstant)
        medelfelen.append(medelfel(lutning, konstant))

kandidater = pd.DataFrame({
    "lutning": lutningar,
    "konstant": konstanter,
    "medelfel": medelfelen
})

print("Antal provade linjer:", len(kandidater))
kandidater.sort_values("medelfel").head(3)

In [ ]:
bast = kandidater.sort_values("medelfel").iloc[0]
bast_lutning = int(bast["lutning"])
bast_konstant = int(bast["konstant"])

rita_linje(bast_lutning, bast_konstant)

### Prova själv

Ändra stegen i `range(-500, 2001, 50)` till `10` i stället för `50`. Hur många linjer provas nu? Blir den bästa linjen mycket bättre? Vad tror du händer med tiden det tar om modellen har tio parametrar i stället för två?

## 6. Det här var maskininlärning

Du har redan gjort de centrala stegen i maskininlärning. Så här heter de:

| Det vi gjorde | ML-begrepp |
|---|---|
| funktionen `linjar_modell` | **modell** |
| `lutning` och `konstant` | **parametrar** |
| `medelfel` | **felmått** |
| loopen som letade upp bästa parametrarna | **träning** |
| storleken vi gissade utifrån | **feature** |
| priset vi ville gissa | **target** |

**Träning betyder att leta upp de parametrar som gör felet på exemplen så litet som möjligt.**

När det vi vill förutsäga är ett tal, som ett pris, kallas det **regression**.

- **Features (`X`)**: information som modellen får använda.
- **Target (`y`)**: det vi vill förutsäga.

Om vi vill förutsäga bostadspris kan exempelvis storlek, antal rum och ålder vara features, medan priset är target.

## 7. Ett problem kvar

Vi valde parametrarna **och** mätte felet på **samma** bostäder. Det är som att få exakt samma frågor på tentan som på övningstentan – ett bra resultat säger inte mycket om hur väl man faktiskt har förstått.

Det vi egentligen vill veta är hur bra modellen gissar på bostäder den **aldrig har sett**. Det tar vi hand om i del 2.

# Del 2: Samma sak med scikit-learn

## 8. `fit()` gör samma sak som vår loop

Vår loop fungerar, men den provar bara ett grovt rutnät av värden och blir mycket långsam när modellen har många parametrar. `LinearRegression` i scikit-learn räknar i stället fram den bästa linjen direkt.

Vi jämför först med loopens resultat, på samma data:

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(housing[["size_m2"]], housing["price_ksek"])

print("scikit-learn:  lutning", round(model.coef_[0], 1), "  konstant", round(model.intercept_))
print("vår for-loop:  lutning", bast_lutning, "  konstant", bast_konstant)

Svaren ligger nära varandra. `fit()` gör alltså samma jobb som vår loop: **anpassar modellens parametrar så att felet på träningsdatan blir litet**.

Att siffrorna inte blir exakt lika har två orsaker: scikit-learn räknar fram svaret exakt i stället för att prova ett rutnät, och `LinearRegression` mäter felet som fel² (kvadrerat fel) i stället för |fel|.

## 9. Välj `X` och `y`

Vi börjar med en enda feature: bostadens storlek.

In [ ]:
X = housing[["size_m2"]]
y = housing["price_ksek"]

print(X.head())
print()
print(y.head())

Dubbla hakparenteser gör `X` till en tvådimensionell DataFrame. Det är den form scikit-learn vanligtvis förväntar sig för features.

## 10. Varför train/test?

Här löser vi problemet från avsnitt 7. Om modellen både tränas och examineras på exakt samma observationer vet vi inte om den kan **generalisera** till nya observationer.

Därför delar vi datan innan vi tränar:

- **training set** – modellen får lära sig från detta,
- **test set** – hålls undan tills vi vill utvärdera modellen.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

print(len(X_train), len(X_test))

## 11. Träna på träningsdatan

I scikit-learn följer många modeller samma mönster:

1. skapa modellen,
2. `fit()` på träningsdata,
3. `predict()` på nya features.

Nu får modellen bara se träningsdelen.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Lutning:", model.coef_[0])
print("Konstant:", model.intercept_)

`model.coef_[0]` motsvarar vår `lutning` och `model.intercept_` vår `konstant`.

## 12. Prediction och ground truth

Nu får modellen features från testdatan. `y_test` är facit, ofta kallat **ground truth**.

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
})

results.head()

### Prova själv

Titta på de första raderna. Verkar prediction och actual ligga nära varandra? Finns några tydliga stora fel?

## 13. Visualisera prediction mot facit

En perfekt modell skulle ligga på diagonalen `predicted = actual`.

In [ ]:
plt.scatter(results["actual"], results["predicted"])
low = min(results.min())
high = max(results.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Facit")
plt.ylabel("Prediction")
plt.title("Prediction mot ground truth")
plt.show()

## 14. MAE på testdatan

Det här är samma felmått som du skrev själv i del 1. Scikit-learn har en färdig funktion, men resultatet blir detsamma:

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE (scikit-learn):  {mae:.1f} tusen kr")
print(f"MAE (som i del 1):   {(y_pred - y_test).abs().mean():.1f} tusen kr")

## 15. Baseline

Ett felmått är svårt att tolka utan jämförelse.

En mycket enkel baseline kan vara:

> Gissa alltid medelvärdet från träningsdatan.

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Modellens MAE:  {mae:.1f}")
print(f"Baselines MAE: {baseline_mae:.1f}")

Om modellen inte slår en enkel baseline behöver vi vara försiktiga med att kalla den användbar.

## 16. Fler features

Nu låter vi modellen använda storlek, antal rum och ålder.

In [ ]:
X = housing[["size_m2", "rooms", "age_years"]]
y = housing["price_ksek"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)

mae_multi = mean_absolute_error(y_test, y_pred_multi)
print(f"MAE med tre features: {mae_multi:.1f}")

Att ge modellen fler features hjälper bara om de faktiskt innehåller relevant information. Fler kolumner är inte automatiskt bättre.

## 17. Modellklassens begränsningar

En linjär modell kan bara representera linjära samband. Vi skapar därför ett U-format samband.

In [ ]:
x_curve = np.linspace(-4, 4, 100)
y_curve = x_curve**2 + rng.normal(0, 1.5, 100)

curve = pd.DataFrame({"x": x_curve, "y": y_curve})

curve_model = LinearRegression()
curve_model.fit(curve[["x"]], curve["y"])
curve["prediction"] = curve_model.predict(curve[["x"]])

plt.scatter(curve["x"], curve["y"], label="data")
plt.plot(curve["x"], curve["prediction"], label="linjär modell")
plt.legend()
plt.show()

Problemet är inte nödvändigtvis "för lite träning". Tänk på loopen i del 1: vi kunde prova hur många lutningar och konstanter som helst, men resultatet blir ändå en rak linje – och en rak linje kan inte bli U-formad.

Det här är en central ML-idé: **modellvalet begränsar vilka samband modellen kan lära sig**.

## 13. Arbetsflödet att memorera

1. Förstå datan.
2. Välj `X` och `y`.
3. Dela train/test.
4. Skapa modell.
5. `fit(X_train, y_train)`.
6. `predict(X_test)`.
7. Utvärdera på `y_test`.
8. Jämför gärna med en baseline.

# Självtest

1. Vad är skillnaden mellan vanlig programmering och maskininlärning?
2. I del 1: vad var modellen, och vilka var dess parametrar?
3. Varför räcker det inte att ta medelvärdet av felen (utan absolutbelopp)?
4. Vad betyder det att "träna" en modell?
5. Vad är skillnaden mellan en feature och target?
6. Varför ska testdatan inte användas för att träna modellen?
7. Vad gör `fit()`?
8. Vad gör `predict()`?
9. Vad är ground truth i vårt exempel?
10. Hur tolkas MAE?
11. Varför använder vi en baseline?
12. Varför kan en linjär modell misslyckas på ett U-format samband även om vi har mycket data?

## Inför laboration 5

Laboration 5 låter dig genomföra hela regressionspipelinen själv.

Se till att du kan:

- skapa `X` och `y` från en DataFrame,
- använda `train_test_split`,
- träna `LinearRegression`,
- skapa predictioner,
- beräkna MAE,
- jämföra modellen med en baseline,
- resonera om vad som händer när fler features används.

# Facit till självtestet

<details>
<summary><strong>Klicka här när du har försökt själv</strong></summary>

1. I vanlig programmering skriver vi reglerna själva. I maskininlärning ger vi exempel (indata och rätt svar) och låter datorn hitta regeln.
2. Modellen var funktionen `linjar_modell`, alltså en rak linje. Parametrarna var `lutning` och `konstant`.
3. Positiva och negativa fel tar ut varandra, så medelvärdet kan bli nära noll även när modellen missar mycket.
4. Att leta upp de parametrar som gör felet på träningsexemplen så litet som möjligt.
5. Features är information modellen får använda; target är det som ska förutsägas.
6. För att testresultatet ska mäta hur modellen fungerar på data den inte tränats på.
7. Anpassar modellens parametrar till träningsdatan – samma jobb som vår loop i del 1, men snabbare och exaktare.
8. Använder den tränade modellen för att göra förutsägelser från features.
9. De verkliga värdena i `y_test`.
10. Genomsnittligt absolut prediktionsfel, i samma enhet som target.
11. För att kontrollera att modellen faktiskt slår en mycket enkel strategi.
12. Därför att modellklassen bara kan representera raka linjära samband.

</details>

## Sammanfattning

En ML-modell är inte magi: den är en funktion med parametrar. Att träna modellen är att leta upp parametrar som ger litet fel på exemplen – något du kunde göra själv med en `for`-loop. Scikit-learn gör samma sak snabbare, och modellen måste sedan **testas på nya observationer**.

Nästa gång byter vi från regression till klassificering och tittar närmare på frågan: **hur vet vi om modellen verkligen är bra?**